<p align="center"><a href="https://www.plus2net.com/python/math-ldexp.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Python math.ldexp(): Multiply Float by Powers of Two (x * 2**i)

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/math-ldexp.php](https://www.plus2net.com/python/math-ldexp.php)

Run cells interactively in Google Colab. Master Python's `math.ldexp()` function, IEEE 754 floating-point binary exponent scaling, exact round-trip reconstruction with `math.frexp()`, overflow/underflow boundaries, and real-world DSP 16-bit audio fixed-point normalization. Try the practice exercise with unit assertions at the end.


### 1. Basic Multiplication by Powers of Two
`math.ldexp(x, i)` calculates `x * (2**i)`. It directly adjusts the IEEE 754 binary exponent field of float `x` by integer `i`.


In [ ]:
import math

print("ldexp(3.0, 4):  ", math.ldexp(3.0, 4))    # 3.0 * 16 = 48.0
print("ldexp(1.5, 3):  ", math.ldexp(1.5, 3))    # 1.5 * 8 = 12.0
print("ldexp(10.0, -2):", math.ldexp(10.0, -2))  # 10.0 / 4 = 2.5
print("ldexp(1.0, -3): ", math.ldexp(1.0, -3))   # 1.0 / 8 = 0.125
print("ldexp(7.25, 0): ", math.ldexp(7.25, 0))   # 7.25 * 1 = 7.25
print("ldexp(-2.5, 2): ", math.ldexp(-2.5, 2))   # -2.5 * 4 = -10.0
print("ldexp(-0.0, 5): ", math.ldexp(-0.0, 5))   # Preserves signed -0.0


### 2. Round-Trip Reconstruction with math.frexp()
`math.ldexp()` is the exact inverse of `math.frexp()`. Decomposing any real number with `frexp()` and passing the mantissa and exponent to `ldexp()` reconstructs the original number with bit-level exactness.


In [ ]:
import math

test_values = [0.0, 0.125, 1.0, 13.375, 1024.0, -88.5, 1e15]
print(f"{'Original (x)':>14} | {'Mantissa (m)':>12} | {'Exp (e)':>7} | {'Reconstructed':>14} | Match?")
print("-" * 64)
for orig in test_values:
    m, e = math.frexp(orig)
    reconstructed = math.ldexp(m, e)
    matches = (reconstructed == orig)
    assert matches, f"Mismatch for {orig}"
    print(f"{orig:14.3f} | {m:12.6f} | {e:7d} | {reconstructed:14.3f} | {matches}")
print("All round-trip reconstructions matched perfectly!")


### 3. Bit-Exact Exponent Scaling vs. Other Methods
Compared to `x * (2 ** i)` or `x * math.pow(2, i)`, `math.ldexp()` directly updates the 11-bit exponent field of the float in memory, avoiding intermediate integer allocation or transcendental power routines.


In [ ]:
import math

x = 1.337
i = 10

via_ldexp = math.ldexp(x, i)
via_pow_operator = x * (2 ** i)
via_math_pow = x * math.pow(2, i)

print("via ldexp:       ", via_ldexp)
print("via 2 ** i:      ", via_pow_operator)
print("via math.pow:    ", via_math_pow)
print("ldexp == 2 ** i: ", via_ldexp == via_pow_operator)


### 4. Error Handling & Floating-Point Boundaries
The exponent parameter `i` must be an integer. Exponents exceeding IEEE 754 limits raise `OverflowError`, while extremely negative exponents gracefully underflow to `0.0`.


In [ ]:
import math

# 1. TypeError on float exponent
try:
    math.ldexp(4.0, 2.5)
except TypeError as err:
    print("Caught expected TypeError:   ", err)

# 2. OverflowError on exponent > 1023
try:
    math.ldexp(1.0, 1024)
except OverflowError as err:
    print("Caught expected OverflowError:", err)

# 3. Underflow to zero
subnormal = math.ldexp(1.0, -1074)
zero_val = math.ldexp(1.0, -1075)
print("Smallest subnormal (exp=-1074):", subnormal)
print("Underflow to zero  (exp=-1075):", zero_val)


### 5. Hands-on Practice Exercise: DSP Audio Fixed-Point Normalizer
In digital audio processing, 16-bit PCM audio samples are stored as signed integers in `[-32768, 32767]`. Audio algorithms require normalized floating-point samples in `[-1.0, 1.0]`.

We use `math.ldexp(sample, -15)` to normalize to float and `math.ldexp(sample, 15)` to quantize back to 16-bit integer PCM.


In [ ]:
import math

def pcm16_to_float(pcm_val):
    """Converts 16-bit signed integer PCM (-32768 to 32767) to normalized float [-1.0, 1.0]."""
    if not isinstance(pcm_val, int):
        raise TypeError("PCM sample must be an integer.")
    if pcm_val < -32768 or pcm_val > 32767:
        raise ValueError("PCM sample out of 16-bit signed range [-32768, 32767].")
    return round(math.ldexp(pcm_val, -15), 6)

def float_to_pcm16(float_val):
    """Converts normalized float audio [-1.0, 1.0] back to 16-bit signed integer PCM."""
    if not isinstance(float_val, (int, float)):
        raise TypeError("Normalized sample must be numeric.")
    clamped = max(-1.0, min(1.0, float(float_val)))
    scaled = math.ldexp(clamped, 15)
    return int(round(scaled))

# Unit Assertions
assert pcm16_to_float(0) == 0.0
assert pcm16_to_float(32767) == 0.999969
assert pcm16_to_float(-32768) == -1.0
assert pcm16_to_float(16384) == 0.5

assert float_to_pcm16(0.0) == 0
assert float_to_pcm16(0.5) == 16384
assert float_to_pcm16(-1.0) == -32768

# Boundary and type checking assertions
try:
    pcm16_to_float(40000)
    assert False, "Should have raised ValueError"
except ValueError:
    pass

raw_audio = [-32768, -16384, 0, 16384, 32767]
normalized = [pcm16_to_float(s) for s in raw_audio]
print("Raw 16-bit PCM: ", raw_audio)
print("Normalized float:", normalized)
print("All DSP audio conversion assertions passed successfully!")


---
### Documentation & Further Reading
* [Python math.ldexp() Tutorial on plus2net](https://www.plus2net.com/python/math-ldexp.php)
* [Python math.frexp() Guide](https://www.plus2net.com/python/math-frexp.php)
* [Python math.modf() Guide](https://www.plus2net.com/python/math-modf.php)
* [Python math.pow() Guide](https://www.plus2net.com/python/math-pow.php)
* [Python Math Module Reference](https://www.plus2net.com/python/math.php)
